# 01 · Alignment (stage A)
Trains the student so its query vectors match the teacher's cached query vectors (no teacher forward pass). Output: `aligned/` (weights, cfg.json, train_log.jsonl). Retrieval is evaluated in 00, not here.

Run 00 cells 1–2 first. Run with `DEBUG = True` first.

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import torch

DEBUG = True    # first pass: ~10k passages, Qwen3-Embedding-0.6B as debug teacher, 3 steps per loop, shape printing
TINY = False    # local CPU smoke run: tiny random models + fake rows, no downloads
NPROCS = torch.cuda.device_count() or 2   # DDP processes (2 CPU processes with gloo when there is no GPU)

# --- paths: each mode gets its own cache + runs dirs, so a debug pass never overwrites a real run
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "tiny" if TINY else "debug" if DEBUG else "full"
PATHS = dict(cache_dir=f"{ROOT}/cache/{MODE}", runs_dir=f"{ROOT}/runs/{MODE}")
PATHS.update(data_path=f"{PATHS['cache_dir']}/data.json",       # built by 00, cell 1
             emb_dir=f"{PATHS['cache_dir']}/emb/teacher")         # built by 00, cell 2

# --- models
MODELS = dict(
    teacher="tiny:1" if TINY else "Qwen/Qwen3-Embedding-0.6B" if DEBUG else "Qwen/Qwen3-Embedding-4B",
    student="tiny:0" if TINY else "Qwen/Qwen3-Embedding-0.6B",
)
DIM = 32 if TINY else 256   # MRL dim shared by teacher index and student (tiny model has hidden 64)

ALIGN_CFG = dict(
    # model
    init=MODELS["student"],           # hub id or checkpoint dir
    dim=DIM, max_query_len=32,
    trainable="all_but_embeddings",   # all | all_but_embeddings (token embeddings frozen -> ~0.44B trainable)
    # loss (see losses.py)
    loss="l2", loss_kwargs={},        # l2 | cosine | hybrid(terms={...})
    # optimisation
    lr=2e-5, lr_schedule="cosine",    # cosine | constant
    warmup_ratio=0.05, min_lr_ratio=0.1,
    epochs=1, batch_size=128,         # batch_size is per GPU
    accum_steps=1, max_grad_norm=1.0,
    amp=True,                         # fp16 autocast + GradScaler; False = fp32 (use if you see NaNs)
    # run
    seed=0, val_every=200, save_dtype="fp16",
    debug=DEBUG,
    # io
    data_path=PATHS["data_path"], emb_dir=PATHS["emb_dir"], out_dir=f"{PATHS['runs_dir']}/aligned",
)
if TINY:
    ALIGN_CFG.update(batch_size=4, max_query_len=16, amp=False)
RUN_SCALING = True
SCALING_STEPS = 3 if (DEBUG or TINY) else 20

## Plan
Resolved config, effective batch, optimizer updates, trainable/total params, first-batch shapes. Trains nothing.

In [ ]:
from ddp import launch
from train import align_main, benchmark_scaling

launch(align_main, {**ALIGN_CFG, "plan_only": True}, nprocs=NPROCS)

## Scaling: 1 vs 2 processes
Samples/s and scaling efficiency (warm-up steps discarded, no checkpoints).

In [ ]:
if RUN_SCALING and NPROCS > 1:
    benchmark_scaling(align_main, ALIGN_CFG, steps=SCALING_STEPS, nprocs_list=(1, NPROCS))

## Debug run
3 steps per GPU, tensor shapes, grad check, weights-in-sync check. Saves to a separate dir.

In [ ]:
launch(align_main, {**ALIGN_CFG, "debug": True, "out_dir": f"{PATHS['runs_dir']}/debug_aligned"}, nprocs=NPROCS)

## Real run

In [ ]:
launch(align_main, ALIGN_CFG, nprocs=NPROCS)

In [ ]:
# what was saved
for out in (ALIGN_CFG["out_dir"],):
    print(out)
    for f in sorted(os.listdir(out)):
        print(f"  {os.path.getsize(os.path.join(out, f)) / 1e6:10.2f} MB  {f}")
    with open(os.path.join(out, "train_log.jsonl")) as fh:
        print("  last log:", fh.readlines()[-1].strip())
print("Next: re-run the last cell of 00_baseline.ipynb to evaluate these checkpoints.")